# Method: measuring league strength from transfers

*Question → Intuition → Math → Code → Assumptions → How it breaks*

## 1. Question

A player's season is scored against the other players in **his** league. So a
2.0 in Ligue 1 and a 2.0 in the Premier League are both "two standard deviations
above your peers" — but the peers are not equally good.

If we pool a career across leagues, as we must when players transfer, how do we
avoid rewarding whoever played in the weakest division?

## 2. Intuition

**A player who changes league is the same footballer on both sides of the
move.** So whatever happens to his score across that move is a measurement of
the difference between the two leagues.

One transfer is a noisy measurement. Thousands of transfers, forming a connected
graph over 25 years, pin down the whole system — the same device that makes
chess ratings comparable across separate rating pools.

## 3. Math

For a move from league $a$ to league $b$:

$$\Delta z = \alpha + \beta \cdot \text{age} + \lambda_a - \lambda_b + \varepsilon$$

- $\lambda$ are the league strengths we want, identified only **up to a
  constant**, so one league is pinned at zero
- $\alpha$ is a global adaptation term: settling into a new league costs
  something on average, and that cost is not league strength
- $\beta$ controls for age, since players move at different career stages and
  decline would otherwise be misread as league difficulty

Fitted by weighted least squares, weighting each move by the smaller of the two
seasons' minutes.

In [ ]:
import warnings

import matplotlib
import pandas as pd

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
offsets = pd.read_parquet(f"{SAMPLE}/league_offsets.parquet")
keepers = pd.read_parquet(f"{SAMPLE}/keeper_ranking.parquet")

summary = (
    offsets.groupby("league")
    .agg(offset=("offset", "mean"), moves=("moves", "sum"))
    .sort_values("offset", ascending=False)
)
summary.round(3)

The Premier League is pinned at zero as the reference. Everything else is
measured against it, in the same units as the player scores.

In [ ]:
offsets["era"] = offsets["season"].str[:2].astype(int) // 5
era = offsets.pivot_table(index="league", columns="era", values="offset", aggfunc="mean")
era.columns = ["2000-04", "2005-09", "2010-14", "2015-19", "2020-24"]
era.round(2)

## 4. Code — and the corroboration that matters

Look at the era table above. The gaps are small in 2000-04 and widen sharply
from 2005 onward.

That is the Premier League's financial ascent — and **nothing about money, TV
deals or transfer fees is in this model.** It was recovered purely from players
changing league and their scores changing with them. When an estimate reproduces
a known historical pattern it did not have access to, that is real evidence the
method works.

In [ ]:
rank_by_moves = offsets.groupby("league")["moves"].sum().sort_values(ascending=False)
print("transfers backing each estimate:")
for lg, n in rank_by_moves.items():
    print(f"  {lg:<22} {n:>6,}")
print("\nAn offset backed by 200 moves deserves less trust than one backed by 3,000,")
print("which is why the count is published alongside the estimate.")

## 5. Assumptions

1. **A transferring player is unchanged by the move**, apart from age and a
   common adaptation effect. Injuries, motivation and tactical fit all violate
   this individually; the hope is they average out.
2. **Transfers are not selective in a way that correlates with the gap.** They
   almost certainly are — players usually move *up* when they excel and *down*
   when they decline, which is exactly the kind of selection that biases this.
3. **League strength is constant within a five-season block.** A compromise:
   per-season offsets would be badly identified from the handful of moves some
   pairs see in one year.

## 6. How it breaks

The model has a failure mode that is invisible unless you look for it:
**with transfers in only one direction between two leagues, the adaptation term
and the league offset are perfectly collinear.**

Both apply to every move. Only their *sign behaviour* separates them — the
offset flips when the direction flips, adaptation does not. If everybody moves
one way, no amount of data separates them, and the solver will split the
difference arbitrarily.

In [ ]:
from gambeta import bridge, kit

cfg = kit.load()


def moves_between(gap, n, both_ways):
    rows = []
    for i in range(n):
        rows += [
            {
                "player_id": f"o{i}",
                "league": "FRA-Ligue 1",
                "season": "0102",
                "score": 1.0 + gap,
                "minutes": 3000,
                "age": 25.0,
            },
            {
                "player_id": f"o{i}",
                "league": "ENG-Premier League",
                "season": "0203",
                "score": 1.0,
                "minutes": 3000,
                "age": 26.0,
            },
        ]
        if both_ways:
            rows += [
                {
                    "player_id": f"i{i}",
                    "league": "ENG-Premier League",
                    "season": "0102",
                    "score": 0.5,
                    "minutes": 3000,
                    "age": 25.0,
                },
                {
                    "player_id": f"i{i}",
                    "league": "FRA-Ligue 1",
                    "season": "0203",
                    "score": 0.5 + gap,
                    "minutes": 3000,
                    "age": 26.0,
                },
            ]
    return pd.DataFrame(rows)


pair = ["ENG-Premier League", "FRA-Ligue 1"]
for both in (True, False):
    m = bridge.find_moves(moves_between(0.5, 40, both))
    o = bridge.solve_offsets(m, cfg, leagues=pair)
    est = o[(o["league"] == "FRA-Ligue 1") & (o["season"] == "0102")]["offset"].item()
    label = "both directions" if both else "one direction only"
    print(f"  {label:<20} true gap -0.50, estimated {est:+.2f}")

With traffic both ways the true gap is recovered. With one-way traffic, half of
it is silently absorbed by the adaptation term and the league looks stronger
than it is.

Real transfer data flows both ways between all four leagues, which is what makes
the estimates identifiable. But a league with mostly outbound moves — a selling
league — would be systematically mis-measured, and that is a live risk rather
than a hypothetical one.